# 机械臂标定

## 导入依赖

In [1]:
import math
from arm5dof_uservo import Arm5DoFUServo
from config import *

## 机械臂初始化

In [2]:
arm = Arm5DoFUServo(is_init_pose=False)

## 设置舵机为阻尼模式

In [3]:
# 设置为阻尼模式
# 数值越大，阻尼越大
arm.set_damping(1000) 

## 样本采集

将机械臂掰直, 水平于桌面, 夹爪张开。 采集舵机原始角度

![](image/pose1.jpg)

In [4]:
pose1_servo_raw_angle = arm.get_servo_angle_list()
# 打印舵机原始角度
j1_0, j2_n90, j3_90, j4_n90, j5_0, j6_90 = pose1_servo_raw_angle
print(f"J1: {j1_0:.1f} J2: {j2_n90:.1f} J3: {j3_90:.1f} J4: {j4_n90:.1f} J5: {j5_0:.1f} J6: {j6_90:.1f}")    

J1: -2.9 J2: 0.9 J3: -94.1 J4: 87.5 J5: -6.5 J6: 91.2


夹爪闭合, 令夹爪运动到机械臂左侧，夹爪末端垂直于桌面，且连杆之间均为90度直角。
![](./image/pose2.jpg)

In [5]:
pose2_servo_raw_angle = arm.get_servo_angle_list()
# 打印舵机原始角度
j1_90, j2_0, j3_0, j4_0, j5_90, j6_0 = pose2_servo_raw_angle
print(f"J1: {j1_90:.1f} J2: {j2_0:.1f} J3: {j3_0:.1f} J4: {j4_0:.1f} J5: {j5_90:.1f} J6: {j6_0:.1f}")    

J1: -93.7 J2: 91.4 J3: -5.1 J4: 1.6 J5: 78.3 J6: 6.6


## 标定舵机

计算各个关节的比例系数与角度偏移量. 用一个简单的一次函数来表示舵机原始角度与机械臂关节弧度之间的关系

$$
angle_i = k_i*\theta_{i} + b_i
$$

* $\theta_i$关节弧度
* $angle_i$ 舵机原始角度
* $k_i$ 比例系数
* $b_i$ 偏移量

In [6]:
def calc_kb(angle_a, angle_b, theta_a, theta_b):
    k = (angle_a-angle_b) / (theta_a-theta_b)
    b = angle_a - k*theta_a
    return k, b

In [7]:
k1, b1 = calc_kb(j1_0, j1_90, 0, math.pi/2)
k2, b2 = calc_kb(j2_n90, j2_0, -math.pi/2, 0)
k3, b3 = calc_kb(j3_90, j3_0, math.pi/2, 0)
k4, b4 = calc_kb(j4_n90, j4_0, -math.pi/2, 0)
k5, b5 = calc_kb(j5_0, j5_90, 0, math.pi/2)
k6, b6 = calc_kb(j6_90, j6_0, math.pi/2, 0)

print(f'JOINT2SERVO_K=[{k1:.3f}, {k2:.3f}, {k3:.3f}, {k4:.3f}, {k5:.3f}, {k6:.3f}]')
print(f"JOINT2SERVO_B=[{b1:.3f}, {b2:.3f}, {b3:.3f}, {b4:.3f}, {b5:.3f}, {b6:.3f}]")

JOINT2SERVO_K=[-57.805, 57.614, -56.659, -54.686, 53.985, 53.858]
JOINT2SERVO_B=[-2.900, 91.400, -5.100, 1.600, -6.500, 6.600]


将打印出来的字符串替换掉`config.py`里面的`JOINT2SERVO_K`与`JOINT2SERVO_B`